# Adaptive Risk Fusion Comparison

Compare Weighted, Dynamic, Logistic Regression, Random Forest, Fuzzy, and Attention fusion strategies for explainable IDS risk scoring.

## 1) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Install Dependencies

In [ ]:
!pip install -q scikit-learn matplotlib seaborn pandas numpy torch

## 3) Import Modules

In [ ]:
import os
import sys
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

# Update this to your repo root in Drive if needed
REPO_PATH = '/content/drive/MyDrive/Deep Learning Project/AI Agentic'
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

from risk_fusion import FusionFactory
from risk_fusion.utils import severity_to_index

## 4) Load Dataset

In [ ]:
import os

path = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed'
print(os.listdir(path))

X_train = np.load(os.path.join(path, 'X_train.npy'))
X_test = np.load(os.path.join(path, 'X_test.npy'))
y_train = np.load(os.path.join(path, 'y_train.npy')).reshape(-1)
y_test = np.load(os.path.join(path, 'y_test.npy')).reshape(-1)

if X_train.ndim == 3 and X_train.shape[-1] == 1:
    X_train = X_train[..., 0]
if X_test.ndim == 3 and X_test.shape[-1] == 1:
    X_test = X_test[..., 0]

print('X_train:', X_train.shape, 'X_test:', X_test.shape)

## 5) Generate Pipeline Signals (confidence, memory_similarity, graph_similarity, fg_strength)

In [ ]:
def minmax(v):
    v = np.asarray(v, dtype=np.float32)
    vmin, vmax = float(v.min()), float(v.max())
    if np.isclose(vmin, vmax):
        return np.full_like(v, 0.5, dtype=np.float32)
    return (v - vmin) / (vmax - vmin)

def simulate_signals(X, y, seed):
    rng = np.random.default_rng(seed)
    abs_mean = minmax(np.mean(np.abs(X), axis=1))
    std = minmax(np.std(X, axis=1))
    y_norm = minmax(y.astype(np.float32))
    attack_flag = (y > 0).astype(np.float32)

    confidence = np.clip(0.50*abs_mean + 0.35*attack_flag + 0.15*y_norm + rng.normal(0, 0.04, len(y)), 0, 1)
    memory_similarity = np.clip(0.45*confidence + 0.30*y_norm + 0.25*abs_mean + rng.normal(0, 0.05, len(y)), 0, 1)
    graph_similarity = np.clip(0.40*confidence + 0.35*std + 0.25*y_norm + rng.normal(0, 0.05, len(y)), 0, 1)
    fg_strength = np.clip(0.55*abs_mean + 0.30*std + 0.15*confidence + rng.normal(0, 0.04, len(y)), 0, 1)

    return np.column_stack([confidence, memory_similarity, graph_similarity, fg_strength]).astype(np.float32)

def risk_to_sev_index(r):
    if r > 0.8:
        return 3
    if r > 0.6:
        return 2
    if r > 0.4:
        return 1
    return 0

def build_targets(signals, y, seed):
    rng = np.random.default_rng(seed + 1)
    c, m, g, f = signals[:, 0], signals[:, 1], signals[:, 2], signals[:, 3]
    attack_bias = (y > 0).astype(np.float32)
    risk = np.clip(0.42*c + 0.24*m + 0.20*g + 0.14*f + 0.08*attack_bias + rng.normal(0, 0.03, len(y)), 0, 1)
    return np.array([risk_to_sev_index(v) for v in risk], dtype=np.int32)

train_signals = simulate_signals(X_train, y_train, RNG_SEED)
test_signals = simulate_signals(X_test, y_test, RNG_SEED + 10)
y_train_sev = build_targets(train_signals, y_train, RNG_SEED)
y_test_sev = build_targets(test_signals, y_test, RNG_SEED + 10)

print(train_signals.shape, test_signals.shape)

## 6) Train Fusion Models

In [ ]:
methods = [
    ('Weighted', 'weighted'),
    ('Dynamic', 'dynamic_weighted'),
    ('LogisticRegression', 'logistic_regression'),
    ('RandomForest', 'random_forest'),
    ('Fuzzy', 'fuzzy_logic'),
    ('Attention', 'attention'),
]

fusions = {}
for name, key in methods:
    model = FusionFactory.create(key)
    if hasattr(model, 'fit'):
        model.fit(train_signals, y_train_sev)
    fusions[name] = model

print('Training complete.')

## 7) Evaluate Models

In [ ]:
rows = []
pred_cache = {}

for name, _ in methods:
    model = fusions[name]
    y_pred = []
    for c, m, g, f in test_signals:
        out = model.compute_risk(float(c), float(m), float(g), float(f))
        y_pred.append(severity_to_index(out['severity']))

    y_pred = np.array(y_pred, dtype=np.int32)
    pred_cache[name] = y_pred

    rows.append({
        'Method': name,
        'Accuracy': accuracy_score(y_test_sev, y_pred),
        'Precision': precision_score(y_test_sev, y_pred, average='macro', zero_division=0),
        'Recall': recall_score(y_test_sev, y_pred, average='macro', zero_division=0),
        'F1': f1_score(y_test_sev, y_pred, average='macro', zero_division=0),
    })

results = pd.DataFrame(rows).sort_values(by='F1', ascending=False).reset_index(drop=True)

## 8) Comparison Table

In [ ]:
results.style.format({'Accuracy': '{:.4f}', 'Precision': '{:.4f}', 'Recall': '{:.4f}', 'F1': '{:.4f}'})

## 9) Plots: Accuracy, F1, Confusion Matrices

In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(data=results, x='Method', y='Accuracy', palette='Blues_d')
plt.ylim(0, 1)
plt.title('Accuracy Comparison')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.barplot(data=results, x='Method', y='F1', palette='Greens_d')
plt.ylim(0, 1)
plt.title('F1 Comparison')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
labels = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
n_methods = len(methods)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, _) in enumerate(methods):
    cm = confusion_matrix(y_test_sev, pred_cache[name], labels=[0, 1, 2, 3])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', cbar=False, ax=axes[idx], xticklabels=labels, yticklabels=labels)
    axes[idx].set_title(name)
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('True')

plt.tight_layout()
plt.show()

## 10) Best Performing Method

In [ ]:
best = results.iloc[0]
print(f"Best Fusion Method: {best['Method']}")
print(f"Accuracy: {best['Accuracy']:.4f}")
print(f"F1 Score: {best['F1']:.4f}")